# Attention and Asset Price Dynamics in Financial Time Series

This project analyzes the relationship between investor attention, proxied by Google Trends data, and financial market behavior. Specifically, it examines whether attention signals and shocks contain predictive information about future returns and volatility across selected assets.

**Author:** Atharva S  
**Date:** April 22 2026

## Imports

In [2]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from pytrends.request import TrendReq
import matplotlib.pyplot as plt
import statsmodels as sm
import os

## 2 & 3: Fetching Data and Data Processing

### Price/ETF data

In [2]:
tickers = ['SPY','BTC-USD','TSLA']
asset_data = yf.download(tickers, start = '2021-01-01')
asset_data.columns = ['_'.join(col)for col in asset_data.columns]
asset_data = asset_data.dropna()  # remove weekends

# create features
for ticker in tickers:
    asset_data[ticker+'_log_ret'] = np.log( asset_data['Close_'+ticker] / asset_data['Close_'+ticker].shift(1) )
    asset_data[ticker+'_log_ret_t+1'] = asset_data[ticker+'_log_ret'].shift(-1)
    asset_data[ticker+'_vol_10'] = asset_data[ticker+'_log_ret'].rolling(window=10).std()
    asset_data[ticker+'_vol_10_t+1'] = asset_data[ticker+'_vol_10'].shift(-1)
    asset_data[ticker+'_shock_10'] = asset_data[ticker+'_log_ret'] / asset_data[ticker+'_vol_10']
    asset_data[ticker+'_shock_10_t+1'] = asset_data[ticker+'_shock_10'].shift(-1)
    asset_data = asset_data.drop(columns=[i+'_'+ticker for i in ['Open','Close','High','Low','Volume']])  # remove unnecessary columns

asset_data = asset_data.dropna()
asset_data.to_csv('asset_data.csv')  # save to csv for reuse
asset_data

[*********************100%***********************]  3 of 3 completed


,SPY_log_ret,SPY_log_ret_t+1,SPY_vol_10,SPY_vol_10_t+1,SPY_shock_10,SPY_shock_10_t+1,BTC-USD_log_ret,BTC-USD_log_ret_t+1,BTC-USD_vol_10,BTC-USD_vol_10_t+1,BTC-USD_shock_10,BTC-USD_shock_10_t+1,TSLA_log_ret,TSLA_log_ret_t+1,TSLA_vol_10,TSLA_vol_10_t+1,TSLA_shock_10,TSLA_shock_10_t+1
Date,,,,,,,,,,,,,,,,,,
2021-01-19,0.007821,0.013744,0.007015,0.007772,1.114907,1.768391,-0.020731,-0.014579,0.075489,0.073785,-0.274621,-0.197589,0.022015,0.006962,0.047317,0.047323,0.465273,0.147110
2021-01-20,0.013744,0.000911,0.007772,0.007746,1.768391,0.117666,-0.014579,-0.142528,0.073785,0.081614,-0.197589,-1.746364,0.006962,-0.006441,0.047323,0.047491,0.147110,-0.135624
2021-01-21,0.000911,-0.003546,0.007746,0.006707,0.117666,-0.528734,-0.142528,0.068333,0.081614,0.081784,-1.746364,0.835536,-0.006441,0.001951,0.047491,0.041582,-0.135624,0.046914
2021-01-22,-0.003546,0.003936,0.006707,0.006593,-0.528734,0.597046,0.068333,-0.019562,0.081784,0.079627,0.835536,-0.245665,0.001951,0.039555,0.041582,0.035845,0.046914,1.103501
2021-01-25,0.003936,-0.001562,0.006593,0.006116,0.597046,-0.255374,-0.019562,0.006266,0.079627,0.069009,-0.245665,0.090806,0.039555,0.002597,0.035845,0.021645,1.103501,0.119960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,0.007860,0.002454,0.007517,0.007673,1.045692,0.319860,0.008370,0.004629,0.018932,0.018590,0.442093,0.248991,0.073431,-0.007812,0.035053,0.034516,2.094864,-0.226335
2026-04-16,0.002454,0.012013,0.007673,0.007520,0.319860,1.597582,0.004629,0.025937,0.018590,0.016400,0.248991,1.581502,-0.007812,0.029691,0.034516,0.028728,-0.226335,1.033524
2026-04-17,0.012013,-0.002002,0.007520,0.008118,1.597582,-0.246586,0.025937,-0.016397,0.016400,0.018055,1.581502,-0.908166,0.029691,-0.020477,0.028728,0.028567,1.033524,-0.716793


### Attention Data

In [ ]:
pytrends = TrendReq()
SPY_kws = ['stock market', 'stock market crash','inflation','interest rates','recession','oil prices','crude oil','Fed rate hike','economy']
BTC_kws = ['bitcoin price','crypto crash','crypto','cryptocurrency','buy bitcoin','bitcoin ETF','inflation hedge','risk off','altcoin','memecoin', 'bitcoin mining']
TSLA_kws = ['tesla stock','elon musk','self driving car','electric vehicles','AI stocks','tesla crash','battery technology','lithium price']
kws = SPY_kws + BTC_kws + TSLA_kws

start_date = pd.to_datetime('2021-01-01')
end_date = pd.to_datetime('2026-04-19')
window_months = 6
overlap_months = 2
attention_data = None

current_start = start_date
current_end = current_start

while current_end < end_date:

    current_end = current_start+pd.DateOffset(months=window_months)

    if current_end > end_date:
        current_end = end_date
    
    timeframe = f'{current_start.date()} {current_end.date()}'
    print(f'\nFetching window {timeframe}')
    window_df = pd.DataFrame()


    for kw in kws:
        print(f'\rFetching {kw:<30}',end='',flush=True)

        fetched = False
        attempts = 0

        while (not fetched) and (attempts < 5):
            try:
                attempts += 1
                pytrends.build_payload(
                    kw_list = [kw],
                    timeframe = timeframe
                )
                data = pytrends.interest_over_time()
                fetched = True
            except Exception as e:
                print(f'Attempt {attempts} failed: {e}')
                time.sleep(60)  # wait 60s for rate limit

        data = data.drop(columns=['isPartial'], errors='ignore')  # remove unnecessary column
        window_df[kw] = data[kw]

    # Scale values to match previous window

    if attention_data is None:
        attention_data = window_df.copy()
    else:
        # get overlap index
        overlap = attention_data.index.intersection(window_df.index)

        # calculate scale factors
        scale_factors = {}
        for kw in kws:
            old_vals = attention_data.loc[overlap, kw]
            new_vals = window_df.loc[overlap, kw]

            scale = old_vals.mean() / new_vals.mean()

            scale_factors[kw]=scale

        # scale each kw
        for kw in kws:
            window_df[kw] = window_df[kw] * scale_factors[kw]
    
    # get non-overlapping part
    window_df = window_df.loc[~window_df.index.isin(attention_data.index)]
    attention_data = pd.concat([attention_data,window_df])

    current_start = current_end - pd.DateOffset(months=overlap_months)

attention_data.to_csv('raw_attention_data.csv')
attention_data


Fetching window 2021-01-01 2021-07-01
Fetching stock market                  Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2021-05-01 2021-11-01
Fetching tesla crash                   Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2021-09-01 2022-03-01
Fetching recession                     Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2022-01-01 2022-07-01
Fetching stock market                  

C:\Users\athar\AppData\Local\Temp\ipykernel_32864\4268836348.py:64: RuntimeWarning: invalid value encountered in scalar divide
  scale = old_vals.mean() / new_vals.mean()


Fetching lithium price                 
Fetching window 2022-05-01 2022-11-01
Fetching lithium price                 
Fetching window 2022-09-01 2023-03-01
Fetching lithium price                 
Fetching window 2023-01-01 2023-07-01
Fetching stock market                  

C:\Users\athar\AppData\Local\Temp\ipykernel_32864\4268836348.py:64: RuntimeWarning: invalid value encountered in scalar divide
  scale = old_vals.mean() / new_vals.mean()


Fetching tesla stock                   Attempt 1 failed: The request failed: Google returned a response with code 429


In [14]:
### Create attention data features
attention_data = pd.read_csv('raw_attention_data.csv', index_col='date')
for kw in kws:
    attention_data[kw + '_diff'] = attention_data[kw].diff()
    attention_data[kw+'_shock_10'] = (attention_data[kw]-attention_data[kw].rolling(window=10).mean()) / attention_data[kw].rolling(window=10).std()

attention_data.to_csv('attention_data.csv')
attention_data

,stock market,stock market crash,inflation,interest rates,recession,oil prices,crude oil,Fed rate hike,economy,bitcoin price,...,electric vehicles_diff,electric vehicles_shock_10,AI stocks_diff,AI stocks_shock_10,tesla crash_diff,tesla crash_shock_10,battery technology_diff,battery technology_shock_10,lithium price_diff,lithium price_shock_10
date,,,,,,,,,,,,,,,,,,,,,
2021-01-01,23.000000,21.000000,24.000000,73.000000,46.000000,43.000000,27.000000,0.0,52.000000,26.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-02,14.000000,17.000000,25.000000,77.000000,53.000000,41.000000,21.000000,0.0,59.000000,39.000000,...,15.000000,NaN,-2.000000,NaN,1.000000,NaN,-1.000000,NaN,4.000000,NaN
2021-01-03,13.000000,18.000000,25.000000,70.000000,52.000000,38.000000,26.000000,0.0,57.000000,47.000000,...,-6.000000,NaN,33.000000,NaN,0.000000,NaN,15.000000,NaN,3.000000,NaN
2021-01-04,30.000000,25.000000,29.000000,79.000000,61.000000,69.000000,51.000000,0.0,73.000000,45.000000,...,-1.000000,NaN,-29.000000,NaN,1.000000,NaN,-8.000000,NaN,23.000000,NaN
2021-01-05,29.000000,26.000000,31.000000,82.000000,68.000000,78.000000,60.000000,0.0,83.000000,37.000000,...,3.000000,NaN,4.000000,NaN,-1.000000,NaN,14.000000,NaN,10.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,27.516094,19.511732,76.735079,125.220601,200.626434,499.871713,432.654950,NaN,146.644168,13.096976,...,16.639382,-0.214981,105.216585,-0.239597,0.341181,-0.644075,6.931544,-0.355062,4.971191,-0.080739
2026-04-16,29.044766,22.056741,78.927510,122.374678,163.776681,477.150272,370.847100,NaN,168.959585,13.970108,...,58.237839,1.189282,368.258046,0.553388,0.255886,-0.417405,20.794632,-0.047541,39.769528,0.558786
2026-04-17,38.216798,22.905077,83.312372,130.912446,167.871098,886.136219,571.722612,NaN,172.147502,15.716371,...,-54.077993,0.318193,683.907800,2.384378,2.985338,2.354518,27.726176,1.840980,-4.971191,1.452069


### Allign Data

In [5]:
asset_data = pd.read_csv('asset_data.csv')
attention_data = pd.read_csv('attention_data.csv')
data = pd.concat([asset_data,attention_data],axis=1)
data = data.dropna()
data.to_csv('data.csv')
data

,Date,SPY_log_ret,SPY_log_ret_t+1,SPY_vol_10,SPY_vol_10_t+1,SPY_shock_10,SPY_shock_10_t+1,BTC-USD_log_ret,BTC-USD_log_ret_t+1,BTC-USD_vol_10,...,electric vehicles_diff,electric vehicles_shock_10,AI stocks_diff,AI stocks_shock_10,tesla crash_diff,tesla crash_shock_10,battery technology_diff,battery technology_shock_10,lithium price_diff,lithium price_shock_10
0,2021-01-19,0.007821,0.013744,0.007015,0.007772,1.114907,1.768391,-0.020731,-0.014579,0.075489,...,1.000000,-0.384008,10.000000,0.541119,-2.000000,-0.149858,-4.0,-0.542354,3.000000,0.903370
1,2021-01-20,0.013744,0.000911,0.007772,0.007746,1.768391,0.117666,-0.014579,-0.142528,0.073785,...,-1.000000,-0.264847,2.000000,0.314874,0.000000,-0.283981,-3.0,-0.537073,-6.000000,0.409558
2,2021-01-21,0.000911,-0.003546,0.007746,0.006707,0.117666,-0.528734,-0.142528,0.068333,0.081614,...,-2.000000,-0.441799,0.000000,0.309931,0.000000,-0.349066,-2.0,-0.693334,-13.000000,-0.969813
3,2021-01-22,-0.003546,0.003936,0.006707,0.006593,-0.528734,0.597046,0.068333,-0.019562,0.081784,...,-10.000000,-1.805917,0.000000,0.312320,0.000000,-0.327630,7.0,0.036882,-11.000000,-1.729427
4,2021-01-25,0.003936,-0.001562,0.006593,0.006116,0.597046,-0.255374,-0.019562,0.006266,0.079627,...,9.000000,-0.123206,-13.000000,-1.612175,-1.000000,-0.781160,0.0,0.387218,31.000000,0.866248
5,2021-01-26,-0.001562,-0.024744,0.006116,0.010292,-0.255374,-2.404153,0.006266,-0.067874,0.069009,...,2.000000,0.342693,13.000000,0.763662,0.000000,-0.705151,10.0,1.509092,-9.000000,0.120772
6,2021-01-27,-0.024744,0.008563,0.010292,0.010696,-2.404153,0.800616,-0.067874,0.095020,0.070572,...,-1.000000,0.172453,5.000000,1.115508,0.000000,-0.456535,6.0,1.760823,1.000000,0.302272
7,2021-01-28,0.008563,-0.020223,0.010696,0.012380,0.800616,-1.633566,0.095020,0.025090,0.070519,...,-2.000000,-0.864826,0.000000,-1.083660,-2.000000,-1.267731,6.0,-0.277466,4.000000,0.330527
8,2021-01-29,-0.020223,0.016509,0.012380,0.013533,-1.633566,1.219874,0.025090,-0.022968,0.068651,...,10.000000,1.118515,26.000000,0.364508,1.000000,0.000000,-5.0,-0.408208,0.000000,0.416168
9,2021-02-01,0.016509,0.014041,0.013533,0.014058,1.219874,0.998788,-0.022968,0.057168,0.066638,...,3.000000,1.612189,-2.000000,0.258258,0.000000,0.000000,2.0,0.087139,-2.000000,0.200505


## 4: Exploratory Data Analysis

In [6]:
data = pd.read_csv('data.csv')
kws_map = {
    'SPY_kws': SPY_kws,
    'BTC-USD_kws': BTC_kws,
    'TSLA_kws': TSLA_kws
}

os.makedirs('plots',exist_ok=True)

for ticker in tickers:
    for kw in kws_map[ticker+'_kws']:
        fig,axes = plt.subplots(3,3,figsize=(12,10),sharex=True) # generate 3x3 subplot of for each ticker kw pair
        axes=axes.flatten()
        i=0

        for ticker_stat in ['_log_ret','_vol_10','_shock_10']:
            for kw_stat in ['','_diff','_shock_10']:
                ax = axes[i]
                ax.plot(data.index,(data[ticker+ticker_stat]-data[ticker+ticker_stat].min()) / (data[ticker+ticker_stat].max()-data[ticker+ticker_stat].min()),label=ticker+ticker_stat) 
                ax.plot(data.index,(data[kw+kw_stat] - data[kw+kw_stat].min()) / (data[kw+kw_stat].max() - data[kw+kw_stat].min()),label=kw+kw_stat)  # scale by min-max for visibility
                ax.set_title(f'{ticker+ticker_stat} vs {kw+kw_stat}')
                ax.legend(fontsize=8)
                i = i+1
        
        plt.tight_layout()
        path = 'plots/' + ticker + '_'+kw+'.png'
        plt.savefig(path,dpi=300)
        plt.close()

## 5: Modeling